# 5 · Bonus lab: a simple LangChain tool-using agent | 30–45 additional minutes
**Optional, after the three-hour core workshop.** Understand an LLM, two tools, and the agent's tool-selection loop. Reuse the RAG pipeline you already built.

Unlike simple RAG, an **agent chooses which tool to use** for each question:
- `support_ticket_metrics` returns **actual SQL aggregates** from your team table.
- `support_kb_search` retrieves **documents with pgvector** (and OpenAI embeddings).

The LLM cannot send arbitrary SQL or execute Python code. The SQL tool only supports three predefined, read-only queries. This protects your shared Supabase instance.

Suggested test prompts:
- "How many tickets do we have by type?"
- "How do I troubleshoot Bluetooth pairing?"
- "Which issue type is most common, and what troubleshooting guidance is available for it?"

## Why this notebook matters
In Notebook 3 the pipeline was **fixed**: every question was embedded, searched and answered from articles. But "How many tickets do we have by type?" is not a question for articles: it needs a **number from the database**. And a question like "Which type is most common, and how do I help those customers?" needs **both**. An **agent** solves this by letting the LLM look at the question and **decide which tool to call**, then use the result to write the answer.

## The big picture
![RAG follows fixed steps: question, retrieve, generate, answer. An agent lets the LLM choose between the metrics tool, the knowledge-base tool or no tool, and tool results go back to the LLM](https://raw.githubusercontent.com/Ngxba/fpt-ghc-data-ai-workshop-level1/main/assets/nb5_rag_vs_agent.png)

| Step | What we do |
|---|---|
| 0 | Install, connect, create the OpenAI client |
| 1 | Reuse the RAG functions from Notebook 3 |
| 2 | Define two **tools** and try them directly, without any LLM |
| 3 | Create the **agent**: model + tools + instructions |
| 4 | **TODO 1:** a question that needs the metrics tool |
| 5 | Inspect what the agent did: which tool, which arguments |
| 6 | **TODO 2:** a question that needs both tools |
| 7 | Discussion: safety, accuracy and when *not* to use an agent |

## Vocabulary you will meet
| Term | Meaning |
|---|---|
| **LLM** | The language model (here `gpt-4.1-mini`) that reads text and writes text. |
| **Tool** | A normal Python function that the LLM is allowed to ask for. The LLM never runs it: **your code does**. |
| **Agent** | An LLM that can use tools, in a loop, until it can answer. |
| **Tool call** | The LLM's request: "run tool X with arguments Y". |
| **Message types** | `HumanMessage` (the user), `AIMessage` (the model, possibly with tool calls), `ToolMessage` (a tool's result). |
| **System prompt** | Standing instructions for the agent: its role and rules. |
| **Guardrail** | A limit built into the tools or prompt (here: fixed queries) so mistakes cannot do damage. |
| **Temperature** | How random the model is. `0` makes answers as repeatable as possible. |

## Step 0 · Setup
**Install:** `langchain` is the framework that runs the agent loop; `langchain-openai` connects it to OpenAI models; the other packages are the same as before (`psycopg` for the database, `pandas` for tables, `openai` for embeddings, `python-dotenv` for your `.env` file).

In [ ]:
%pip install -q python-dotenv "langchain>=1,<2" "langchain-openai>=1,<2" "psycopg[binary]>=3.2,<4" "pandas>=2.2,<3" "openai>=1.55,<3"

## Step 0 · Colab secrets (Colab only)
**What:** on Google Colab, load your credentials from **Secrets** (the key icon in the left sidebar), so you enter them once for all notebooks instead of typing them in each one. For every name below click **Add new secret**, paste the value, and switch **Notebook access** on. On local Jupyter skip this: your `.env` file is used.

| Secret name | Value |
|---|---|
| `LAB_PROJECT_REF` | Supabase project ref, given by your instructor |
| `LAB_POOLER_HOST` | Transaction pooler host (host name only, not the full URL) |
| `LAB_TEAM_NUMBER` | Your team number, e.g. `1` |
| `LAB_DB_PASSWORD` | Your team database password |
| `OPENAI_API_KEY` | Your own budgeted OpenAI key (never the instructor key) |

**What you should see:** nothing printed. If a secret is missing, the cell lists it and the next cells ask you to type it.

In [ ]:
# Colab only: copy your Colab Secrets into environment variables.
# Outside Colab (local Jupyter) this cell does nothing and your .env file / the prompts are used.
import os
try:
    from google.colab import userdata
except ImportError:
    userdata = None

if userdata is not None:
    missing = []
    for name in ["LAB_PROJECT_REF", "LAB_POOLER_HOST", "LAB_TEAM_NUMBER", "LAB_DB_PASSWORD", "OPENAI_API_KEY"]:
        try:
            os.environ[name] = userdata.get(name)
        except Exception:  # secret not created, or "Notebook access" is switched off
            missing.append(name)
    if missing:
        print("Not found in Colab Secrets (you will be asked to type them):", ", ".join(missing))

In [ ]:
import os, getpass
from dotenv import load_dotenv
load_dotenv()
from contextlib import closing
import psycopg
from psycopg import sql
import pandas as pd

# Reenter these in EACH notebook. Use the team credential supplied privately by your instructor.
PROJECT_REF = os.getenv("LAB_PROJECT_REF") or input("Supabase project ref: ").strip()
POOLER_HOST = os.getenv("LAB_POOLER_HOST") or input("Transaction pooler host (not full URL): ").strip()
TEAM_NUMBER = int(os.getenv("LAB_TEAM_NUMBER") or input("Team number (e.g. 1): ").strip())
TEAM_SCHEMA = f"team_{TEAM_NUMBER:02d}"
DB_PASSWORD = os.getenv("LAB_DB_PASSWORD") or getpass.getpass("Team database password: ")
DB_USER = f"lab_team_{TEAM_NUMBER:02d}.{PROJECT_REF}"

def connect():
    # Supabase Free transaction pooler; psycopg prepared statements disabled.
    return psycopg.connect(host=POOLER_HOST, port=6543, dbname="postgres",
                           user=DB_USER, password=DB_PASSWORD, sslmode="require",
                           connect_timeout=12, prepare_threshold=None)

def fetch_df(query, params=None):
    with closing(connect()) as conn:
        with conn.cursor() as cur:
            cur.execute(query, params)
            if cur.description is None: return pd.DataFrame()
            return pd.DataFrame(cur.fetchall(), columns=[x.name for x in cur.description])

def run_sql(query, params=None):
    with closing(connect()) as conn:
        with conn.transaction():
            with conn.cursor() as cur:
                cur.execute(query, params)

print("Connected to:", TEAM_SCHEMA)
print(fetch_df("SELECT current_user AS user_name, current_database() AS db_name"))

**Connect to the database:** the next cell is the same provided helper as in Notebooks 1 to 3 (`connect`, `fetch_df`, `run_sql`). **What you should see:** `Connected to: team_XX` and your database user.

In [ ]:
import os, getpass
from openai import OpenAI
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or getpass.getpass("Your budgeted OpenAI API key (never instructor master key): ")
client = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-3-small"      # 1536 dimensions by default
CHAT_MODEL = "gpt-4.1-mini"

def embed(text):
    return client.embeddings.create(model=EMBED_MODEL, input=text).data[0].embedding

def vector_literal(vector):
    assert len(vector) == 1536, "Embedding dimension must match vector(1536)"
    return "[" + ",".join(str(float(x)) for x in vector) + "]"

**Create the OpenAI client:** the cell after it reads your **own budgeted** API key (never the instructor's master key), creates the client, and defines two small helpers from Notebook 3: `embed(text)` (text to a 1,536-number vector) and `vector_literal(...)` (format the vector for PostgreSQL). Both are used by the knowledge-base tool.

## Step 1 · Reuse your RAG functions from Notebook 3
This provided cell contains two functions you already know:
- **`retrieve_articles(question, k=3)`** = RAG steps 4 to 6: embed the question, search the shared and team articles by cosine distance (`<=>`), return the `k` most similar. It will become the **knowledge-base tool**.
- **`grounded_answer(question, passages)`** = RAG step 8: build the prompt and ask the LLM to answer only from the articles. In this notebook we keep it only to **compare** with the agent at the end.

Nothing new happens here: the point is that an agent does not replace your RAG code, it **reuses it as a tool**.

In [ ]:
def retrieve_articles(question, k=3):
    q_vec = vector_literal(embed(question))
    # Instructor-provisioned views/tables; schema is constructed only from a numeric team ID.
    query = f"""
      SELECT article_id, title, content, 1 - (embedding OPERATOR(extensions.<=>) %s::extensions.vector) AS similarity
      FROM (
         SELECT article_id, title, content, embedding FROM shared.knowledge_articles
         UNION ALL
         SELECT article_id, title, content, embedding FROM {TEAM_SCHEMA}.knowledge_articles
      ) kb
      WHERE embedding IS NOT NULL
      ORDER BY embedding OPERATOR(extensions.<=>) %s::extensions.vector
      LIMIT %s
    """
    return fetch_df(query, (q_vec,q_vec,int(k)))

def grounded_answer(question, passages):
    context = "\n\n".join(
        f"[{row.article_id}] {row.title}: {row.content}" for row in passages.itertuples()
    )
    response = client.responses.create(
        model=CHAT_MODEL,
        input=[
          {"role":"system","content": "Answer only from retrieved knowledge articles. Cite the article IDs in square brackets. If context does not answer the question, say you do not know. Do not invent policies."},
          {"role":"user","content":f"Question: {question}\n\nRetrieved articles:\n{context}"},
        ],
    )
    return response.output_text

**What you should see:** no output, the functions are just defined.

## Step 2 · Tools: giving the LLM two safe abilities
![Anatomy of a LangChain tool: decorator, name and arguments, docstring, allowlist of fixed queries, guard and result, plus what the LLM actually receives](https://raw.githubusercontent.com/Ngxba/fpt-ghc-data-ai-workshop-level1/main/assets/nb5_tool_anatomy.png)

A **tool** is an ordinary Python function with a decorator, `@tool`. LangChain turns three things into the description the LLM sees: the **function name**, its **arguments** (type hints) and its **docstring**. That means the **docstring is the LLM's only guide** to when to use the tool, so write it clearly.

### 2.1 · `support_kb_search`: documents

In [ ]:
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

@tool
def support_kb_search(question: str) -> str:
    """Search official workshop support KB articles for troubleshooting or policy guidance; return article IDs."""
    hits = retrieve_articles(question, k=3)
    return "\n".join(
        f"[{r.article_id}] {r.title}: {r.content}" for r in hits.itertuples()
    ) or "No relevant articles found"

This tool takes a free-text `question`, calls `retrieve_articles` and returns the top 3 articles as text lines like `[KB-001] Title: content`. If nothing is found it returns a short message. The agent will cite those `[article_id]`s.

### 2.2 · `support_ticket_metrics`: numbers
This tool takes a `metric` name and returns **database aggregates as JSON**.
- `choices` is a dictionary of **three fixed SQL queries**, written by you: `summary` (total, closed, average satisfaction), `by_type` and `by_channel` (ticket counts).
- The LLM chooses only the **key**. It cannot write SQL.
- `if metric not in choices` refuses anything else with a message.
- `fetch_df(...).to_json(orient="records")` turns the result table into text, because a tool must return text the LLM can read.

In [ ]:
@tool
def support_ticket_metrics(metric: str) -> str:
    """Get structured ticket analytics. metric must be: 'summary', 'by_type', or 'by_channel'. No arbitrary SQL."""
    choices = {
        "summary": f"""SELECT COUNT(*) AS tickets,
            COUNT(*) FILTER (WHERE status='Closed') AS closed,
            ROUND(AVG(satisfaction_rating),2) AS avg_satisfaction
            FROM {TEAM_SCHEMA}.clean_tickets""",
        "by_type": f"""SELECT ticket_type, COUNT(*) AS tickets
            FROM {TEAM_SCHEMA}.clean_tickets GROUP BY ticket_type ORDER BY tickets DESC""",
        "by_channel": f"""SELECT channel, COUNT(*) AS tickets
            FROM {TEAM_SCHEMA}.clean_tickets GROUP BY channel ORDER BY tickets DESC""",
    }
    if metric not in choices:
        return "Unsupported metric; choose summary, by_type, or by_channel."
    return fetch_df(choices[metric]).to_json(orient="records")

### 2.3 · Try the tools directly (no LLM involved)
Before putting a model in charge, test each tool **yourself**. A decorated tool has an `.invoke({...})` method that takes a dictionary of arguments. We call it three times: a valid metric, an **unsafe** value, and a knowledge-base search.

In [ ]:
print("by_type   ->", support_ticket_metrics.invoke({"metric": "by_type"}))
print("unsafe    ->", support_ticket_metrics.invoke({"metric": "DROP TABLE clean_tickets"}))
print("kb search ->", support_kb_search.invoke({"question": "Bluetooth pairing"})[:300], "...")

**What you should see**
- `by_type` → JSON such as `[{"ticket_type":"Technical issue","tickets":132},{"ticket_type":"Cancellation request","tickets":124}, ...]` (five types, same counts as your SQL in Notebooks 1 and 2).
- `unsafe` → `Unsupported metric; choose summary, by_type, or by_channel.` The "attack" is simply refused: no SQL was run.
- `kb search` → up to 3 lines starting with `[KB-...]`; if you completed Notebook 3, your `TEAM-XX-001` Bluetooth article should be among them.

![Guardrails: the LLM can only pick one of three allowed metric names, anything else is refused; the knowledge-base tool passes the text as a parameter into a fixed SQL query](https://raw.githubusercontent.com/Ngxba/fpt-ghc-data-ai-workshop-level1/main/assets/nb5_guardrails.png)

**Why this is safe:** for the metrics tool the LLM can only pick one of three names. For the knowledge-base tool the text is passed as a **parameter** (`%s`) into a fixed query, so it cannot change the SQL. The LLM **chooses**, your code **executes**.

## Step 3 · Create the agent
The next cell builds the agent from three ingredients:
- **`ChatOpenAI(model="gpt-4.1-mini", temperature=0)`**: the LLM. `temperature=0` makes it as repeatable as possible, which matters when you compare runs.
- **`tools=[support_kb_search, support_ticket_metrics]`**: the abilities it may use.
- **`system_prompt`**: the standing rules. Read it: it tells the agent to use the metrics tool for numbers, the KB tool for troubleshooting or policy, to **cite article IDs**, to never invent policies or data, and to explain limits if data is not available. The system prompt and the tool docstrings together decide the agent's behaviour.

`create_agent(...)` returns an object whose `.invoke({...})` runs the whole loop for you: ask the LLM, run the tool it requests, give it the result, repeat, until it answers.

In [ ]:
model = ChatOpenAI(model="gpt-4.1-mini", temperature=0, api_key=OPENAI_API_KEY)
agent = create_agent(
    model=model,
    tools=[support_kb_search, support_ticket_metrics],
    system_prompt=(
        "You are a support analytics agent. Use support_ticket_metrics for numeric/count questions "
        "and support_kb_search for troubleshooting/policy questions. Cite KB article IDs when used. "
        "Do not invent policies or data. You cannot execute arbitrary SQL or mutate data. "
        "If data is not available, explain the limitation."
    ),
)
print("Agent ready: 2 tools registered")

**What you should see:** `Agent ready: 2 tools registered`.

## Step 4 · TODO 1: ask the agent to pick a tool
Write a question about **tickets by type** (a counting question, *not* a knowledge-base question) and run it.

**How the call works**
- `agent.invoke({"messages": [{"role": "user", "content": prompt}]})` sends a conversation with one user message.
- It returns a dictionary; `result["messages"]` is the **whole conversation** after the agent finished, including the tool calls.
- `result["messages"][-1].content` is the **last message**: the final answer.

*Hint: the system prompt says numeric and count questions use `support_ticket_metrics`. Ask for counts, for example "by type" or "by channel".*

In [ ]:
# TODO 1: Ask a metric question (not a knowledge-base question)
prompt = "How many support tickets do we have by type?"
result = agent.invoke({"messages":[{"role":"user","content":prompt}]})
print("Agent answer:",result["messages"][-1].content)

**What you should see:** a sentence or list with the ticket counts per type (for example Technical issue 132, Cancellation request 124, Refund request 123, Billing inquiry 112, Product inquiry 109). The agent never saw your SQL: it picked the tool and wrote the answer from the JSON.

## Step 5 · Explore decisions: what was called?
Don't assume which tool was used: **look**. A single `invoke` hides a small loop:

![One agent call: the LLM decides whether a tool is needed, the tool call request is run by your code, the result goes back to the LLM, and the messages list records every step](https://raw.githubusercontent.com/Ngxba/fpt-ghc-data-ai-workshop-level1/main/assets/nb5_agent_loop.png)

`result["messages"]` records every step: the user's message, the `AIMessage` with the **tool call request**, the `ToolMessage` with the tool's **result**, and the final `AIMessage`. The next cell walks through them: an `AIMessage` with `tool_calls` shows the tools the model asked for; a `ToolMessage` shows what came back (the first 450 characters).

In [ ]:
for msg in result["messages"]:
    role=type(msg).__name__
    if getattr(msg,"tool_calls",None):
        print(role,"requested tools:",[t["name"] for t in msg.tool_calls])
    elif role=="ToolMessage":
        print("Tool returned:", str(msg.content)[:450])
print("Now try a troubleshooting question and observe a different tool.")

**What you should see:** `AIMessage requested tools: ['support_ticket_metrics']`, then `Tool returned: [{"ticket_type": ...` (and the agent's final answer is the last message, not printed here). That is proof the agent chose the metrics tool.

### 5.2 · See the arguments, and a different tool
The tool *name* is only half of the story: the **arguments** show what the model decided to pass. This cell asks a troubleshooting question and prints each tool request with its arguments, the tool result, and the final answer.

In [ ]:
kb_result = agent.invoke({"messages": [{"role": "user", "content": "How do I troubleshoot Bluetooth pairing?"}]})
for msg in kb_result["messages"]:
    if getattr(msg, "tool_calls", None):
        print("LLM requested:", [(t["name"], t["args"]) for t in msg.tool_calls])
    elif type(msg).__name__ == "ToolMessage":
        print("Tool returned:", str(msg.content)[:200], "...")
print()
print("Final answer:", kb_result["messages"][-1].content)

**What you should see:** `LLM requested: [('support_kb_search', {'question': ...})]`: a **different tool** this time, with the question text as the argument. Then the articles, and a final answer that cites article IDs such as `[KB-...]` or your `[TEAM-XX-001]`. The same agent, a different decision, only because the question was different.

## Step 6 · TODO 2: a question that needs BOTH tools
Now write a question that **no single tool can answer**. It must need a count from the data **and** troubleshooting guidance from the articles, for example "which ticket type is most common, and what guidance helps with it?".

**What to expect:** the agent should first call `support_ticket_metrics` to find the most common type, then `support_kb_search` with a question about that type, then combine both results. Models do not always behave the same way, so **check the calls instead of assuming**.

In [ ]:
# TODO 2: Write a question that requires ticket counts AND an article-based solution.
both_prompt="Which ticket type is most common, and what KB guidance helps with it?"
result2=agent.invoke({"messages":[{"role":"user","content":both_prompt}]})
print(result2["messages"][-1].content)

**What you should see:** one answer that **states the most common type with its count** (Technical issue, 132 tickets in the reference data) **and** gives guidance **citing article IDs**.

### 6.2 · Check which tools were used
Reuse the same loop to see the order of the calls. Did the model really use both tools?

In [ ]:
for msg in result2["messages"]:
    if getattr(msg, "tool_calls", None):
        print("LLM requested:", [(t["name"], t["args"]) for t in msg.tool_calls])
    elif type(msg).__name__ == "ToolMessage":
        print("Tool returned:", str(msg.content)[:200], "...")

**What you should see:** at least two `LLM requested:` lines: first `support_ticket_metrics`, then `support_kb_search`, each followed by a `Tool returned:` line. If only one tool was used, rewrite your question so that it clearly needs both a **number** and **guidance**, and run it again.

### 6.3 · Compare with plain RAG (Notebook 3)
For a knowledge question the fixed RAG pipeline gives a similar answer. The difference is **who decides**: RAG always retrieves; the agent chose to. Run both and compare.

In [ ]:
q = "How do I troubleshoot Bluetooth pairing?"
print("--- Plain RAG (always retrieves, then answers) ---")
print(grounded_answer(q, retrieve_articles(q, 3)))
print()
print("--- Agent (decided to call support_kb_search) ---")
print(kb_result["messages"][-1].content)

**What you should see:** two similar answers. Now try the question "How many tickets do we have by type?" with `grounded_answer`: plain RAG has no way to get the counts, because its only knowledge is the articles. That is exactly what the agent adds.

## C. Bonus discussion
- Why do **predefined queries** make an SQL agent safer than arbitrary SQL generation?
- Did the model actually use both tools? Inspect tool calls instead of assuming.
- What can go wrong if retrieved articles are not accurate?
- When is a deterministic pipeline simpler than an agent?

**Not production-ready:** no access-control/auth workflow, approval gates, output validation or comprehensive rate limits. This is a workshop demonstration.

### RAG pipeline or agent?
| | **Fixed RAG pipeline (Notebook 3)** | **Agent (this notebook)** |
|---|---|---|
| Who decides the steps | You, in code | The LLM, at run time |
| Predictable? | Always the same steps | Can differ between runs |
| Cost and speed | One embedding + one LLM call | Two or more LLM calls, plus tools |
| Handles several kinds of question | No, only articles | Yes, picks the right tool |
| Easy to test and debug | Very | Needs trace inspection (Step 5) |
| Use it when | The task is always the same | Questions need different tools or a combination |

### What you did, step by step
| Step | Action | Result |
|---|---|---|
| 1 | Reused `retrieve_articles` | Notebook 3 code becomes a building block |
| 2 | Wrote two tools and tested them without an LLM | You know exactly what each tool returns, and that unsafe input is refused |
| 3 | Created the agent | Model + tools + system prompt |
| 4 | Asked a counting question | The agent chose `support_ticket_metrics` |
| 5 | Inspected the messages | You saw the tool name, its arguments and its result |
| 6 | Asked a mixed question | The agent combined two tools |

### Reflection questions
1. Which sentence in the **system prompt** or tool **docstring** made the agent choose a tool? What would happen if you removed it?
2. The agent returned the right numbers. Where did the numbers come from: the model's memory, or a tool? How do you know?
3. What new tool would help the support team next? What guardrail would it need?